# SECTION 0: Imports

In [0]:
import requests
import json
from pyspark.sql import Row

# SECTION 1: Setup

In [0]:
%run ../config

# SECTION 2: Dataset-specific constants

In [0]:
IUCR_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.iucr_codes"

IUCR_DATASET_ID = "c7ck-438e"
IUCR_ENDPOINT = f"https://{SOCRATA_DOMAIN}/api/v3/views/{IUCR_DATASET_ID}/query.json"
PAGE_SIZE_IUCR = 1000

IUCR_COLUMNS = ["iucr", "primary_description", "secondary_description", "index_code", "active"]

# SECTION 3: Fetch Function

In [0]:
def fetch_iucr():

    app_token = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)

    headers = {
        "X-App-Token": app_token,
        "Content-Type": "application/json"
    }

    payload = {
        "query": "SELECT *",
        "page": {
            "pageNumber": 1,
            "pageSize": PAGE_SIZE_IUCR
        },
        "includeSynthetic": False
    }

    response = requests.post(
        IUCR_ENDPOINT,
        headers=headers,
        json=payload,
        timeout=60
    )
    response.raise_for_status()
    rows = response.json()

    if not rows:
        raise ValueError(f"[ERROR]-[IUCR][INGEST] the API returned 0 rows. {IUCR_TABLE} was NOT overwritten.")
    missing = [c for c in IUCR_COLUMNS if not any(c in row for row in rows)]
    if missing:
        raise ValueError(f"[ERROR]-[IUCR] columns {missing} are not in the API answer (the source changed?). {IUCR_TABLE} was NOT overwritten.")

    return rows

# SECTION 4: Normalize rows into a DataFrame

In [0]:
def normalize_row(row):
    result = {}
    for col in IUCR_COLUMNS:
        value = row.get(col)
        if value is None:
            result[col] = None
        elif isinstance(value, bool):
            result[col] = "true" if value else "false"
        else:
            result[col] = str(value)
    return result

In [0]:
raw_rows = fetch_iucr()
normalized_rows = [normalize_row(row) for row in raw_rows]

In [0]:
iucr_schema = ", ".join(f"{c} string" for c in IUCR_COLUMNS)
df = spark.createDataFrame([Row(**row) for row in normalized_rows], iucr_schema)

# SECTION 5: Write to Bronze

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(IUCR_TABLE)
print(f"Loaded {df.count()} rows into {IUCR_TABLE}")